In [ ]:
from datetime import datetime
import os
import pickle
import numpy as np

from generation.surface_generation import generate_surfaces, SimulationConfig
from models.framework import DeepONet, Trainer

# ============================================================
# 1️⃣ Generate synthetic Rough Bergomi IV surfaces
# ============================================================

cfg = SimulationConfig(M=50000, n=500, T_max=2.0, S0=1.0, G=5)
surfaces = generate_surfaces(
    num_sets=1,
    forward_curves_per_set=10,
    cfg=cfg,
    seed=42,
    randomize_grid=True,
)

# ============================================================
# 2️⃣ Save generated data (full + compact)
# ============================================================

# Create data directory if missing
day_folder = f"data/{datetime.now().strftime('%Y-%m-%d')}"
os.makedirs(day_folder, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
pkl_file = f"{day_folder}/surfaces_{timestamp}.pkl"

# --- Save full dataset with metadata ---
with open(pkl_file, "wb") as f:
    pickle.dump({
        "cfg": cfg.__dict__,
        "surfaces": surfaces
    }, f)
print(f"Saved full dataset to {pkl_file}")


KeyboardInterrupt: 

In [ ]:
# ============================================================
# 3️⃣ Continue to training
# ============================================================

train_loader, val_loader, branch_dim = DeepONet.prepare_data(surfaces)

model = DeepONet(branch_in_dim=branch_dim, trunk_in_dim=2, latent_dim=64, hidden_dim=64)
model._build_networks()

trainer = Trainer(model, train_loader, val_loader, lr=1e-3)
trainer.fit(epochs=10)


{'set_id': 0,
 'fwd_id': 0,
 'grid_id': 0,
 'params': {'eta': 3,
  'rho': -0.5,
  'H': 0.4,
  'xi0_knots': array([0.05, 0.1 , 0.1 , 0.1 , 0.1 , 0.1 , 0.1 , 0.1 ])},
 'grid': {'strikes': array([0.5       , 0.64008771, 0.6634843 , 0.82636109, 0.85219654,
         1.00142357, 1.14079534, 1.16474879, 1.30896343, 1.44194189,
         1.49696314]),
  'maturities': array([0.15271063, 0.34220762, 0.68485377, 0.96086011, 1.19908042,
         1.43179634, 1.7211105 , 2.        ])},
 'price_surface': array([[0.00000000e+00, 1.91214784e-05, 3.30234716e-05, 1.14395550e-03,
         1.92369160e-03, 3.13852300e-02, 3.65893678e-03, 2.63286470e-03,
         4.94703777e-04, 1.31874366e-04, 7.64944508e-05],
        [1.41164350e-04, 7.87658204e-04, 1.02680826e-03, 5.96482844e-03,
         8.00705723e-03, 4.58416336e-02, 1.26627220e-02, 1.04840166e-02,
         4.21652025e-03, 2.27448115e-03, 1.77086984e-03],
        [2.46826756e-03, 5.79261104e-03, 6.63021414e-03, 1.76795682e-02,
         2.11105703e-02, 6